In [ ]:
import os
import sys


In [ ]:
# @title Authenticate with HuggingFace, skip if you have a HF_TOKEN secret

# Authenticate user for HuggingFace if needed. Enter token below if requested.
from huggingface_hub.utils import HfFolder
from huggingface_hub import login
import os
hf_token = os.getenv("HF_TOKEN")
login(token=hf_token)

In [ ]:
def safe_pip_install(package):
    try:
        # Detect Colab
        in_colab = 'google.colab' in sys.modules

        # Use shell-style install in Colab
        if in_colab:
            print(f"Installing {package} in Colab...")
            # Use !pip or %pip to avoid subprocess errors
            get_ipython().system(f"pip install {package}")
        else:
            print(f"Installing {package} in standard environment...")
            # Use subprocess for non-Colab environments
            import subprocess
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    except Exception as e:
        print(f"Failed to install {package}: {e}")

In [ ]:
# === remove conflicts you don't need for HF training ===
###use this on Runpod
python -m pip uninstall -y transformers==4.41.0 peft accelerate bitsandbytes \
  trl tensorflow tensorflow-decision-forests numba \
  cuml-cu12 cudf-cu12 dask-cuda distributed-ucxx-cu12 umap-learn librosa shap \
  stumpy pynndescent dopamine-rl spacy thinc \
  opencv-python opencv-contrib-python opencv-python-headless albumentations albucore || true

# === clean, compatible HF stack ===
python -m pip install -U pip setuptools wheel
python -m pip install "numpy==2.0.2" pillow
python -m pip install "transformers==4.40.0" "peft==0.11.1" "accelerate==0.29.3" "bitsandbytes==0.43.1"
python -m pip install "opencv-python-headless==4.12.0.88" "albumentations==2.0.8" "albucore==0.0.24"

python -m pip install einops timm sentencepiece

python - << 'PY'
import sys, numpy, transformers, peft
print("NumPy:", numpy.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)
print("If NumPy was already imported earlier in this process, restart Python before training.")
PY

In [ ]:
# ---- Pin to CheXagent-compatible stack ----
%pip -q install -U pip setuptools wheel

# 1) Remove conflicting packages (ignore "not installed" warnings)
%pip -q uninstall -y transformers==4.41.0 peft accelerate bitsandbytes \
  trl tensorflow tensorflow-decision-forests numba \
  cuml-cu12 cudf-cu12 dask-cuda distributed-ucxx-cu12 umap-learn librosa shap \
  stumpy pynndescent dopamine-rl spacy thinc opencv-python opencv-contrib-python opencv-python-headless \
  albumentations albucore || true

%pip -q uninstall -y sentence-transformers tsfresh stumpy \
  dask-cudf-cu12 cudf-cu12 cuml-cu12 dask-cuda distributed-ucxx-cu12 || true

# 2) Install versions that work with CheXagent's remote code
%pip -q install "numpy==2.0.2" pillow
%pip -q install "transformers==4.40.0" "peft==0.11.1" "accelerate==0.29.3" "bitsandbytes==0.43.1"
# CheXagent imports these at import time:
%pip -q install "opencv-python-headless==4.12.0.88" "albumentations==2.0.8" "albucore==0.0.24"

# (Often needed by vision stacks; cheap to include)
%pip -q install einops timm sentencepiece

import numpy, transformers, peft, cv2, albumentations as A, timm, einops
print("NumPy", numpy.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| cv2", cv2.__version__, "| albumentations", A.__version__)
print("Go to Runtime > Restart runtime, then re-run your CheXagent load + fine_tune call.")


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments

tok = AutoTokenizer.from_pretrained("gpt2")
m = AutoModelForCausalLM.from_pretrained("gpt2")

class Tiny(torch.utils.data.Dataset):
    def __len__(self): return 2
    def __getitem__(self, i):
        ids = tok("hi", return_tensors="pt").input_ids[0]
        return {"input_ids": ids, "labels": ids.clone()}

args = TrainingArguments(output_dir="tmp", per_device_train_batch_size=1, max_steps=1, report_to="none")
Trainer(model=m, args=args, train_dataset=Tiny()).train()
print("Trainer works")


In [ ]:
def detect_environment():
    # Check for GPU
    has_gpu = torch.cuda.is_available()

    # Check for Colab-specific environment
    is_colab = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())

    # Check for RunPod-specific environment
    is_runpod = 'RUNPOD_POD_ID' in os.environ or os.path.exists('/workspace')

    if is_runpod and has_gpu:
        return 'runpod'
    elif is_colab and not has_gpu:
        return 'colab'
    else:
        return 'unknown'

In [ ]:
from pathlib import Path

#check the environement
env = detect_environment()
if env == 'runpod':
  ##rclone sync gdrive:/MyDrive/MLProjects/foundation-models-radiology /workspace/MLProjects/foundation-models-radiology
  ROOT = Path('/workspace/MLProjects/foundation-models-radiology')
elif env == 'colab':
  from google.colab import drive
  drive.mount('/content/drive')
  ###once mounted the folders can be referenced
  ROOT = Path('/content/drive/MyDrive/MLProjects/foundation-models-radiology')

else:
  sys.exit("Error: No platform recognised")

DICOM_DIR = ROOT / 'PTXHeadtoHeadSmall'   # use the exact folder name as on Drive
JPEG_DIR = ROOT / 'cxr_jpegs'
JPEG_DIR.mkdir(exist_ok=True)
print("exists:", ROOT.exists())


In [ ]:
import torch
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Union
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

class CheXAgentSimpleDataset(Dataset):
    def __init__(self, records: List[Dict[str, Any]], tokenizer):
        self.recs = records
        self.tok = tokenizer
    def _query_with_image(self, image, text: str) -> str:
      if isinstance(image, str) and image.lower().endswith(".dcm"):
          image = dicom_to_pil(image)  # <- turn DICOM path into a PIL.Image
      return self.tok.from_list_format([{"image": image}, {"text": text}])
    def __len__(self): return len(self.recs)
    def __getitem__(self, i: int) -> Dict[str, torch.Tensor]:
        r = self.recs[i]
        query = self._query_with_image(r["image"], r["instruction"])
        answer = r["answer"]
        messages = [
            {"role": "system", "content": "You are a helpful radiology assistant."},
            {"role": "user", "content": query},
            {"role": "assistant", "content": answer},
        ]
        input_ids = self.tok.apply_chat_template(messages, add_generation_prompt=False, return_tensors="pt")[0]
        prompt_only = self.tok.apply_chat_template(messages[:-1], add_generation_prompt=True, return_tensors="pt")[0]
        labels = input_ids.clone(); labels[: prompt_only.size(0)] = -100
        attention_mask = torch.ones_like(input_ids)
        return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

@dataclass
class DynamicPadCollator:
    pad_id: int
    def __call__(self, batch):
        max_len = max(x["input_ids"].size(0) for x in batch)
        input_ids, labels, attention_mask = [], [], []
        for x in batch:
            ids, labs, mask = x["input_ids"], x["labels"], x["attention_mask"]
            pad = max_len - ids.size(0)
            if pad > 0:
                ids  = torch.cat([ids,  torch.full((pad,), self.pad_id, dtype=torch.long)])
                labs = torch.cat([labs, torch.full((pad,), -100, dtype=torch.long)])
                mask = torch.cat([mask, torch.zeros(pad, dtype=torch.long)])
            input_ids.append(ids); labels.append(labs); attention_mask.append(mask)
        return {"input_ids": torch.stack(input_ids), "labels": torch.stack(labels), "attention_mask": torch.stack(attention_mask)}

def fine_tune_chexagent_lora(
    model, tokenizer, train_records: List[Dict[str, Any]], val_records: Optional[List[Dict[str, Any]]] = None,
    output_dir: str = "chexagent-2-3b-lora", num_train_epochs: int = 1, per_device_train_batch_size: int = 1,
    grad_accum_steps: int = 8, learning_rate: float = 2e-4, lora_r: int = 16, lora_alpha: int = 32,
    lora_dropout: float = 0.05, target_modules: Optional[List[str]] = None,
):
    if target_modules is None:
        target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
    model.config.use_cache = False; model.train()
    model = prepare_model_for_kbit_training(model)
    lora_cfg = LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
                          bias="none", task_type="CAUSAL_LM", target_modules=target_modules)
    model = get_peft_model(model, lora_cfg)

    train_ds = CheXAgentSimpleDataset(train_records, tokenizer)
    val_ds = CheXAgentSimpleDataset(val_records, tokenizer) if val_records else None
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
    collator = DynamicPadCollator(pad_id=pad_id)
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

    args = TrainingArguments(
        output_dir=output_dir, num_train_epochs=num_train_epochs,
        per_device_train_batch_size=per_device_train_batch_size, per_device_eval_batch_size=1,
        gradient_accumulation_steps=grad_accum_steps, gradient_checkpointing=True,
        learning_rate=learning_rate, lr_scheduler_type="cosine", warmup_ratio=0.03,
        logging_steps=10, evaluation_strategy="steps" if val_ds is not None else "no",
        eval_steps=200 if val_ds is not None else None, save_steps=500, report_to="none",
        bf16=use_bf16, ddp_find_unused_parameters=False,
    )
    trainer = Trainer(model=model, args=args, data_collator=collator, train_dataset=train_ds, eval_dataset=val_ds)
    trainer.train()
    model.save_pretrained(output_dir); tokenizer.save_pretrained(output_dir)
    print(f" Saved LoRA adapter to: {output_dir}")
    return model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image


base = "StanfordAIMI/CheXagent-2-3b"
tokenizer = AutoTokenizer.from_pretrained(base, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    base, device_map="auto", trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
)
model.eval()

train_records = [
    {"image": Image.new("L", (512, 512)), "instruction": "What abnormalities are present?", "answer": "No acute cardiopulmonary abnormality."},
    {"image": Image.new("L", (512, 512)), "instruction": "Summarize findings.", "answer": "Normal cardiomediastinal silhouette. No focal consolidation or pleural effusion."},
]
_ = fine_tune_chexagent_lora(model, tokenizer, train_records, val_records=None, output_dir="chexagent-lora")


In [ ]:
!pip install pydicom
import numpy as np
from PIL import Image
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

def dicom_to_pil(
    path,
    *, use_voi_lut: bool = True,
    as_rgb: bool = True,
    invert_monochrome1: bool = True
) -> Image.Image:
    """Load a DICOM into a PIL.Image (8-bit)."""
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array  # decoded into a numpy array (int)

    # Apply VOI LUT/windowing if present
    if use_voi_lut:
        try:
            arr = apply_voi_lut(arr, ds)
        except Exception:
            pass

    # Invert if MONOCHROME1 (black/white reversed)
    if invert_monochrome1 and getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        arr = np.max(arr) - arr

    # Normalize to 0–255 uint8
    arr = arr.astype(np.float32)
    arr -= arr.min()
    maxv = arr.max() if arr.max() > 0 else 1.0
    arr = (arr / maxv * 255.0).clip(0, 255).astype(np.uint8)

    img = Image.fromarray(arr, mode="L")
    if as_rgb:
        img = img.convert("RGB")  # many VLMs expect 3 channels
    return img


In [ ]:
for name, param in model.named_parameters():
    if param.device.type == "meta":
        print(f"{name} is on the meta device")

In [ ]:
# Run on all images and collect responses
import csv
# Define query
question = "Does this chest X-ray show a pneumothorax?"

results = []
for path in jpeg_paths:
    answer = ask_chexagent(path, question)
    results.append((path, answer))
    print(f" {Path(path).name} → {answer}")

with open("chexagent_results.csv", "w") as f:
    writer = csv.writer(f)
    writer.writerow(["Image", "Answer"])
    writer.writerows(results)
